[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/16_hyperparameter_optimization/03_bayesian_optimization_optuna.ipynb)

# Bài 03 — Bayesian Optimization và Optuna

**🎯 Mục tiêu**
- Ý tưởng **Bayesian optimization**: xây **mô hình thay thế** (surrogate) của hàm mục tiêu, dùng **acquisition function** để chọn điểm thử tiếp theo — cân bằng **khai thác** (exploitation) và **khám phá** (exploration).
- Cài đặt BO 1 chiều from scratch với Gaussian Process + **Expected Improvement**.
- Dùng **Optuna** (TPE sampler, pruning, không gian điều kiện) — và so sánh với random search.
- Thực hành tốt: không gian tìm kiếm, ngân sách, tránh overfit validation, tái lập.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/16_hyperparameter_optimization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

from ml_course.visualization import set_style

set_style()

## 1. Bayesian optimization trong 1 chiều

Hàm mục tiêu "đắt" $f(x)$ (giả sử mỗi lần đánh giá mất 1 giờ). Sau vài điểm đã đánh giá:
1. Fit **Gaussian Process** (chương 23) → với mỗi $x$: dự đoán trung bình $\mu(x)$ và độ bất định $\sigma(x)$.
2. Tính **Expected Improvement** — kỳ vọng mức cải thiện so với điểm tốt nhất hiện tại $f^*$:

$$\text{EI}(x) = (\mu - f^* - \xi)\,\Phi(z) + \sigma\,\phi(z), \qquad z = \frac{\mu - f^* - \xi}{\sigma}$$

   EI lớn khi $\mu$ cao (khai thác) **hoặc** $\sigma$ lớn (khám phá).
3. Đánh giá $f$ tại $\arg\max \text{EI}$, lặp lại.

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


def f(x):                                                        # "hàm đắt" — đỉnh toàn cục HẸP ở x ≈ 0.78, đỉnh phụ rộng ở x ≈ 0.5
    return np.sin(3 * x) * np.exp(-0.6 * x) + 0.6 * np.exp(-((x - 0.78) ** 2) / 0.01)


def expected_improvement(mu, sigma, best, xi=0.01):
    sigma = np.maximum(sigma, 1e-9)
    z = (mu - best - xi) / sigma
    return (mu - best - xi) * norm.cdf(z) + sigma * norm.pdf(z)


def make_gp():
    kernel = Matern(length_scale=0.2, length_scale_bounds=(0.01, 2), nu=2.5) + WhiteKernel(1e-6, noise_level_bounds="fixed")
    return GaussianProcessRegressor(kernel, normalize_y=True, random_state=0)


def bayes_opt(n_init=3, n_steps=8, seed=1, record=False):
    r = np.random.default_rng(seed)
    X_obs = r.uniform(0, 2, n_init).reshape(-1, 1)
    y_obs = f(X_obs.ravel())
    snapshots = []
    for step in range(n_steps):
        gp = make_gp().fit(X_obs, y_obs)
        mu, sigma = gp.predict(grid, return_std=True)
        ei = expected_improvement(mu, sigma, y_obs.max())
        x_next = grid[np.argmax(ei)]
        if record:
            snapshots.append((X_obs.copy(), y_obs.copy(), mu, sigma, ei, x_next[0]))
        X_obs = np.vstack([X_obs, x_next])
        y_obs = np.r_[y_obs, f(x_next[0])]
    return X_obs, y_obs, snapshots


grid = np.linspace(0, 2, 500).reshape(-1, 1)
f_max = f(grid.ravel()).max()
X_obs, y_obs, snaps = bayes_opt(seed=1, record=True)

fig, axes = plt.subplots(4, 2, figsize=(14, 13), gridspec_kw={"height_ratios": [2, 1, 2, 1]})
for k, step in enumerate([0, 2, 4, 7]):
    Xs, ys, mu, sigma, ei, x_next = snaps[step]
    ax, ax_ei = axes[2 * (k // 2), k % 2], axes[2 * (k // 2) + 1, k % 2]
    ax.plot(grid, f(grid.ravel()), "k--", lw=1, label="f thật (không biết)")
    ax.plot(grid, mu, "b", label="μ(x) — GP")
    ax.fill_between(grid.ravel(), mu - 2 * sigma, mu + 2 * sigma, alpha=0.2)
    ax.scatter(Xs, ys, color="r", zorder=3, label="đã đánh giá")
    ax.set_title(f"bước {step + 1}: {len(ys)} điểm, tốt nhất = {ys.max():.3f}")
    ax_ei.plot(grid, ei, "g")
    ax_ei.axvline(x_next, color="g", ls=":")
    ax_ei.set_ylabel("EI")
    if k == 0:
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
print(f"sau {len(y_obs)} lần đánh giá: x tốt nhất = {X_obs[np.argmax(y_obs), 0]:.3f}, f = {y_obs.max():.4f} | f max thật = {f_max:.4f}")

Một lần chạy có thể may mắn — so sánh trên nhiều seed, **cùng ngân sách 11 lần đánh giá**:

In [ ]:
bo_success = np.mean([bayes_opt(seed=s)[1].max() > 0.95 * f_max for s in range(20)])
rand_success = np.mean([f(np.random.default_rng(s).uniform(0, 2, 11)).max() > 0.95 * f_max for s in range(500)])
print(f"tìm được f > 95% max sau 11 lần đánh giá: Bayesian optimization {bo_success:.0%} | random search {rand_success:.0%}")

BO tập trung đánh giá ở vùng hứa hẹn và vùng còn chưa chắc chắn, nên tìm được đỉnh hẹp với ít lần đánh giá hơn hẳn random search. Nhược điểm: GP mở rộng kém khi nhiều chiều (> ~20) và nhiều điểm (> vài nghìn); không gian hỗn hợp/điều kiện khó mô hình hoá → **TPE** (Optuna, Hyperopt) là lựa chọn thực tế phổ biến.

## 2. Optuna: TPE, pruning và không gian điều kiện

**TPE** (Tree-structured Parzen Estimator): chia các lần thử đã có thành "tốt" (top $\gamma$%) và "xấu"; ước lượng mật độ $\ell(x)$ của nhóm tốt và $g(x)$ của nhóm xấu; chọn $x$ tối đa $\ell(x)/g(x)$. Xử lý tự nhiên tham số rời rạc, phân loại, **điều kiện** (tham số chỉ tồn tại khi chọn một model cụ thể).

In [ ]:
try:
    import warnings

    warnings.filterwarnings("ignore", message="IProgress not found")
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print("Chưa cài optuna — pip install optuna (phần dưới sẽ được bỏ qua).")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_customer_churn

churn = make_customer_churn(n=3000, seed=10)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
y = churn["churn"].to_numpy()
cv = StratifiedKFold(4, shuffle=True, random_state=0)


def objective(trial):
    model_name = trial.suggest_categorical("model", ["logreg", "rf", "hgb"])             # không gian ĐIỀU KIỆN
    if model_name == "logreg":
        model = make_pipeline(StandardScaler(), LogisticRegression(C=trial.suggest_float("C", 1e-3, 1e2, log=True), max_iter=3000))
    elif model_name == "rf":
        model = RandomForestClassifier(n_estimators=150, max_features=trial.suggest_float("max_features", 0.1, 1.0),
                                       min_samples_leaf=trial.suggest_int("rf_min_leaf", 1, 50, log=True), random_state=0)
    else:
        model = HistGradientBoostingClassifier(learning_rate=trial.suggest_float("lr", 0.01, 0.3, log=True),
                                               max_leaf_nodes=trial.suggest_int("leaves", 4, 64, log=True),
                                               min_samples_leaf=trial.suggest_int("hgb_min_leaf", 5, 200, log=True),
                                               max_iter=200, early_stopping=False, random_state=0)
    return cross_val_score(model, X, y, cv=cv, scoring="roc_auc").mean()


if HAS_OPTUNA:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
    study.optimize(objective, n_trials=30)
    print(f"tốt nhất: ROC-AUC CV = {study.best_value:.4f} |",
          {k: round(v, 4) if isinstance(v, float) else v for k, v in study.best_params.items()})
    trials = study.trials_dataframe()
    print(trials["params_model"].value_counts().to_string())

TPE dần tập trung thử model/vùng tham số cho kết quả tốt — ở đây dồn phần lớn lần thử vào **Logistic Regression**, model tốt nhất trên dữ liệu churn (log-odds gần tuyến tính, chương 12). Lịch sử tối ưu:

In [ ]:
if HAS_OPTUNA:
    values = trials["value"].to_numpy()
    plt.plot(values, "o", alpha=0.5, label="giá trị mỗi lần thử")
    plt.plot(np.maximum.accumulate(values), "r-", label="tốt nhất tới thời điểm đó")
    plt.xlabel("lần thử")
    plt.ylabel("ROC-AUC (CV)")
    plt.legend()
    plt.title("Optuna (TPE)")
    plt.show()

### Pruning: dừng sớm lần thử kém

Báo cáo điểm trung gian (vd sau mỗi fold hoặc mỗi epoch) bằng `trial.report`; pruner (Median, Hyperband) dừng lần thử nếu nó tệ hơn các lần thử trước ở cùng bước — kết hợp ý tưởng successive halving (bài 02) với TPE.

In [ ]:
if HAS_OPTUNA:
    def objective_pruned(trial):
        params = {"learning_rate": trial.suggest_float("lr", 0.005, 0.5, log=True),
                  "max_leaf_nodes": trial.suggest_int("leaves", 2, 128, log=True),
                  "min_samples_leaf": trial.suggest_int("min_leaf", 2, 300, log=True)}
        scores = []
        for step, (tr, va) in enumerate(cv.split(X, y)):
            m = HistGradientBoostingClassifier(max_iter=200, early_stopping=False, random_state=0, **params).fit(X.iloc[tr], y[tr])
            scores.append(roc_auc_score(y[va], m.predict_proba(X.iloc[va])[:, 1]))
            trial.report(np.mean(scores), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return np.mean(scores)

    study_p = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=1),
                                  pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    study_p.optimize(objective_pruned, n_trials=30)
    states = pd.Series([t.state.name for t in study_p.trials]).value_counts()
    print(f"tốt nhất = {study_p.best_value:.4f} | trạng thái các lần thử: {states.to_dict()}")

## 3. Thực hành tốt khi tuning

1. **Baseline trước**: model mặc định + feature tốt thường quan trọng hơn tuning.
2. **Không gian hợp lý**: log-scale cho tham số tỉ lệ; khoảng đủ rộng (kiểm tra tham số tốt nhất có ở biên không).
3. **Ngân sách**: random/TPE 30–100 lần thử thường đủ cho 3–6 tham số; dùng pruning/successive halving khi mỗi lần thử đắt.
4. **Đừng overfit validation**: hàng trăm lần thử trên một tập validation nhỏ → chọn cấu hình "may mắn". Dùng CV, và đánh giá cuối trên test/nested CV.
5. **Tái lập**: cố định seed của sampler và model; lưu toàn bộ lịch sử (Optuna storage, MLflow — chương 39).
6. **Phân tích sau tuning**: tham số nào quan trọng (`optuna.importance`), tương tác — để lần sau tìm nhanh hơn.

| Phương pháp | Khi nào |
|---|---|
| Grid search | ≤ 2 tham số rời rạc, cần bảng kết quả đầy đủ |
| Random search | mặc định tốt, dễ song song |
| Successive halving / Hyperband | mỗi lần thử đắt, có "resource" giảm được (mẫu, epoch) |
| Bayesian (GP) | ít chiều, liên tục, mỗi lần thử rất đắt |
| TPE (Optuna) | không gian hỗn hợp/điều kiện, cần pruning — lựa chọn thực tế phổ biến |

## 🧠 Kiểm tra nhanh

1. Acquisition function cân bằng hai mục tiêu nào?
2. Vì sao EI lớn ở vùng chưa đánh giá dù $\mu$ không cao?
3. Không gian "điều kiện" là gì? Cho ví dụ.
4. Bạn chạy 1 000 lần thử Optuna trên một tập validation 500 mẫu. Rủi ro gì?

## 📌 Tóm tắt

- BO: surrogate (GP/TPE) + acquisition (EI) → chọn điểm thử thông minh, ít lần đánh giá.
- Optuna: TPE, không gian điều kiện, pruning — công cụ thực hành chính.
- Tuning là tối ưu trên validation → vẫn cần đánh giá cuối trung thực.

## 📚 Tài liệu tham khảo

- Snoek, Larochelle & Adams (2012) — *Practical Bayesian Optimization of Machine Learning Algorithms*
- Bergstra et al. (2011) — *Algorithms for Hyper-Parameter Optimization* (TPE)
- Akiba et al. (2019) — *Optuna: A Next-generation Hyperparameter Optimization Framework*